# RQ2 — SATD and Non-SATD Lifecycle Evolution

## Research Question

**How do SATD and non-SATD issues differ in their evolution throughout the Agile development lifecycle?**

## Key Finding

SATD issues have **longer and somewhat more change-intensive observed lifecycles** than non-SATD issues. The clearest difference is temporal: SATD issues remain observable much longer and accumulate more recorded effort. They also experience somewhat more scope, estimation, ownership, review, and testing activity, but most process-count effects are negligible.

## Dataset Overview

Two analysis populations are required because detailed change histories are not available for every project.

- **Overall lifecycle cohort:** 202,213 issues, including 52,677 SATD issues, from 25 projects.
- **Detailed process-history cohort:** 3,662 issues, including 468 SATD issues, from only 3 projects.

## Main Findings

### 1. SATD issues have substantially longer observed lifespans

A censoring-aware Kaplan–Meier analysis estimates median time to recorded resolution of **1,105.18 days** for SATD issues versus **278.52 days** for non-SATD issues (log-rank χ² = 5,791.19, *p* < 0.001). In a project-stratified Cox model adjusted for relative Story Points and missing estimates, SATD classification has a hazard ratio of **0.838** (95% CI: 0.810–0.866, *p* < 0.001), indicating an estimated 16.2% lower instantaneous resolution rate. The Cox estimate uses a reproducible project-by-label stratified sample of 20,379 issues; Kaplan–Meier estimates use the full valid cohort. These are associations with recorded resolution, not causal effects.

### 2. SATD issues accumulate more recorded effort

Among issues with positive effort records, median total effort is **30,430 minutes** for SATD issues and **11,609 minutes** for non-SATD issues (Cliff's δ = 0.157, small; Holm-adjusted *p* < 0.001). This is an unadjusted association because the adjusted effort model did not converge reliably.

### 3. Scope and ownership activity is somewhat higher

SATD issues have a median of **1 scope change versus 0** for non-SATD issues (δ = 0.145) and **2 ownership changes versus 1** (δ = 0.125). Description changes occur in **31.41%** of SATD issues compared with **18.13%** of non-SATD issues. Estimation changes are also more frequent, but their effect is negligible (δ = 0.057).

### 4. SATD issues enter review and testing more often

Review is entered by **20.09%** of SATD issues versus **12.52%** of non-SATD issues. Testing is entered by **7.48%** versus **3.91%**. The largest supported transition difference is Development→Review, observed for **19.02%** of SATD issues and **11.52%** of non-SATD issues. However, no individual stage-duration difference remains significant after Holm correction.

## Overall Conclusion

**SATD issues are distinguished most clearly by a lower recorded-resolution rate and greater recorded effort, with smaller differences in change activity and quality-control-stage participation.** Most process-count effects are negligible, and detailed histories cover only three projects. Therefore, the process and transition findings are exploratory and should not be generalized to all projects. These results show association; they do not demonstrate that long lifecycles or process instability cause SATD.


## Analysis populations and an essential coverage constraint

The issue-level file covers 25 projects, but observable transition/stage histories
exist for only a subset of issues in three projects. Consequently:

- **Overall-duration analyses** use the full issue-level cohort.
- **Change-count, transition, workflow-path, and stage analyses** use only issues
  linked to the transition history.

This prevents zero-filled values from projects without event histories from being
mistaken for genuine absence of lifecycle activity. Process-model confidence
intervals based on only three project clusters are exploratory and should not be
generalized to all 25 projects.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from scipy.stats import chi2_contingency, fisher_exact, kruskal, mannwhitneyu
from statsmodels.stats.multitest import multipletests
import statsmodels.api as sm
import statsmodels.formula.api as smf

OUTPUT_DIR = Path("rq2_results")
OUTPUT_DIR.mkdir(exist_ok=True)
LIFE_PATH = Path("rq2_lifecycle_dataset.csv")
TRANSITION_PATH = Path("rq2_transition_dataset.csv")
STAGE_PATH = Path("rq2_stage_duration_dataset.csv")
for path in [LIFE_PATH, TRANSITION_PATH, STAGE_PATH]:
    if not path.exists():
        raise FileNotFoundError(path)

ALPHA = 0.05
MIN_SATD_TYPE_GROUP = 20
print("Output:", OUTPUT_DIR.resolve())


## 1. Load, clean, and validate the linked datasets


In [ ]:
df = pd.read_csv(LIFE_PATH, low_memory=False)
transitions = pd.read_csv(TRANSITION_PATH, low_memory=False)
stage_df = pd.read_csv(STAGE_PATH, low_memory=False)

required = {
    "lifecycle": {"issue_id", "project_id", "sprint_id", "is_satd"},
    "transitions": {"issue_id", "project_id", "is_satd", "from_stage", "to_stage"},
    "stage": {"issue_id", "project_id", "is_satd", "lifecycle_stage", "observed_minutes_in_stage"},
}
for name, data in [("lifecycle", df), ("transitions", transitions), ("stage", stage_df)]:
    missing = sorted(required[name] - set(data.columns))
    if missing:
        raise ValueError(f"{name} is missing required columns: {missing}")
    data["is_satd"] = pd.to_numeric(data["is_satd"], errors="coerce")
    data.dropna(subset=["is_satd"], inplace=True)
    data["is_satd"] = data["is_satd"].astype(int)
    data.drop(data.index[~data["is_satd"].isin([0, 1])], inplace=True)

if df["issue_id"].duplicated().any():
    raise ValueError(f"Issue-level file contains {df['issue_id'].duplicated().sum()} duplicate issue rows.")

transitions["from_stage"] = transitions["from_stage"].fillna("UNKNOWN").astype(str)
transitions["to_stage"] = transitions["to_stage"].fillna("UNKNOWN").astype(str)
stage_df["lifecycle_stage"] = stage_df["lifecycle_stage"].fillna("UNKNOWN").astype(str)
stage_df["observed_minutes_in_stage"] = pd.to_numeric(
    stage_df["observed_minutes_in_stage"], errors="coerce"
)

life_identity = df.set_index("issue_id")[["project_id", "is_satd"]]
cross_file_rows = []
for name, data in [("transitions", transitions), ("stage", stage_df)]:
    identity = data[["issue_id", "project_id", "is_satd"]].drop_duplicates()
    within_issue_conflicts = identity.groupby("issue_id")[["project_id", "is_satd"]].nunique().gt(1).any(axis=1)
    linked = identity.merge(
        life_identity.reset_index(), on="issue_id", how="left", suffixes=("_file", "_life"),
    )
    cross_file_rows.append({
        "dataset": name, "rows": len(data), "issues": data["issue_id"].nunique(),
        "projects": data["project_id"].nunique(),
        "within_issue_identity_conflicts": int(within_issue_conflicts.sum()),
        "unlinked_issues": int(linked["project_id_life"].isna().sum()),
        "project_mismatches": int((linked["project_id_file"] != linked["project_id_life"]).fillna(False).sum()),
        "label_mismatches": int((linked["is_satd_file"] != linked["is_satd_life"]).fillna(False).sum()),
    })
cross_file_audit = pd.DataFrame(cross_file_rows)
cross_file_audit.to_csv(OUTPUT_DIR / "cross_file_integrity_audit.csv", index=False)
if cross_file_audit[[
    "within_issue_identity_conflicts", "unlinked_issues", "project_mismatches", "label_mismatches"
]].to_numpy().sum() > 0:
    raise ValueError("Cross-file identity or label integrity failed; inspect cross_file_integrity_audit.csv")

history_ids = set(transitions["issue_id"].unique())
process_df = df[df["issue_id"].isin(history_ids)].copy()
stage_issue_ids = set(stage_df["issue_id"].unique())
process_df["has_stage_history"] = process_df["issue_id"].isin(stage_issue_ids)

coverage_by_project = (
    df.assign(has_transition_history=df["issue_id"].isin(history_ids))
    .groupby("project_id")
    .agg(
        lifecycle_issues=("issue_id", "size"), satd_prevalence=("is_satd", "mean"),
        history_issues=("has_transition_history", "sum"),
        history_coverage=("has_transition_history", "mean"),
    ).reset_index()
)
coverage_by_project.to_csv(OUTPUT_DIR / "history_coverage_by_project.csv", index=False)
cohort_audit = pd.DataFrame([
    {"cohort": "all issue-level", "issues": len(df), "projects": df["project_id"].nunique(),
     "satd_issues": int(df["is_satd"].sum()), "satd_prevalence": df["is_satd"].mean()},
    {"cohort": "observable process history", "issues": len(process_df),
     "projects": process_df["project_id"].nunique(), "satd_issues": int(process_df["is_satd"].sum()),
     "satd_prevalence": process_df["is_satd"].mean()},
    {"cohort": "stage-duration rows", "issues": stage_df["issue_id"].nunique(),
     "projects": stage_df["project_id"].nunique(), "satd_issues": stage_df.loc[stage_df["is_satd"].eq(1), "issue_id"].nunique(),
     "satd_prevalence": stage_df.drop_duplicates("issue_id")["is_satd"].mean()},
])
cohort_audit.to_csv(OUTPUT_DIR / "cohort_audit.csv", index=False)
print("Dataset integrity:")
print(cross_file_audit.to_string(index=False))
print("\nAnalysis cohorts:")
print(cohort_audit.to_string(index=False))
print("\nHistory coverage by project:")
print(coverage_by_project.to_string(index=False))


## 2. Lifecycle feature families and reusable statistical functions

Count distributions are compared with Mann-Whitney U and Cliff's delta. Holm
correction controls family-wise error across each collection of tests. Effect size
direction is positive when SATD issues have larger values.


In [ ]:
feature_families = {
    "Scope": [
        "description_change_count", "summary_change_count",
        "acceptance_criteria_change_count", "issue_type_change_count", "scope_change_count",
    ],
    "Estimation": [
        "story_point_change_count", "actual_story_point_change_count",
        "time_estimate_change_count", "estimation_change_count",
    ],
    "Ownership": [
        "assignee_change_count", "team_change_count", "sprint_change_count",
        "priority_change_count", "ownership_change_count",
    ],
    "Workflow": [
        "status_transition_count", "blocked_count", "reopen_count",
        "review_return_count", "testing_return_count", "backward_transition_count",
        "rework_entry_count",
    ],
}
all_count_features = [
    feature for features in feature_families.values() for feature in features
    if feature in process_df.columns
]
feature_to_family = {
    feature: family for family, features in feature_families.items() for feature in features
}
for feature in all_count_features:
    process_df[feature] = pd.to_numeric(process_df[feature], errors="coerce")

def descriptive_numeric(data, variables):
    rows = []
    for variable in variables:
        satd = pd.to_numeric(data.loc[data["is_satd"] == 1, variable], errors="coerce").dropna()
        non_satd = pd.to_numeric(data.loc[data["is_satd"] == 0, variable], errors="coerce").dropna()
        rows.append({
            "variable": variable, "satd_n": len(satd), "non_satd_n": len(non_satd),
            "satd_mean": satd.mean(), "non_satd_mean": non_satd.mean(),
            "satd_median": satd.median(), "non_satd_median": non_satd.median(),
            "satd_q1": satd.quantile(.25), "satd_q3": satd.quantile(.75),
            "non_satd_q1": non_satd.quantile(.25), "non_satd_q3": non_satd.quantile(.75),
        })
    return pd.DataFrame(rows)

def mann_whitney_cliff(data, variable):
    satd = pd.to_numeric(data.loc[data["is_satd"] == 1, variable], errors="coerce").dropna().to_numpy()
    non_satd = pd.to_numeric(data.loc[data["is_satd"] == 0, variable], errors="coerce").dropna().to_numpy()
    if len(satd) == 0 or len(non_satd) == 0:
        return None
    statistic, p_value = mannwhitneyu(satd, non_satd, alternative="two-sided")
    delta = 2 * statistic / (len(satd) * len(non_satd)) - 1
    return {
        "variable": variable, "satd_n": len(satd), "non_satd_n": len(non_satd),
        "satd_median": np.median(satd), "non_satd_median": np.median(non_satd),
        "mann_whitney_u": statistic, "p_value": p_value, "cliffs_delta": delta,
    }

def cliff_interpretation(delta):
    value = abs(delta)
    if value < 0.147: return "Negligible"
    if value < 0.33: return "Small"
    if value < 0.474: return "Medium"
    return "Large"

def add_holm(data, p_col="p_value"):
    result = data.copy()
    result["p_holm"] = np.nan
    valid = result[p_col].notna()
    if valid.any():
        result.loc[valid, "p_holm"] = multipletests(result.loc[valid, p_col], method="holm")[1]
    result["significant_holm"] = result["p_holm"] < ALPHA
    return result


## 3. Scope, estimation, ownership, and workflow evolution

These analyses use the observable-history cohort only. Binary event prevalence
separates whether an event ever occurred from how many times it occurred.


In [ ]:
descriptive_counts = descriptive_numeric(process_df, all_count_features)
descriptive_counts["family"] = descriptive_counts["variable"].map(feature_to_family)
descriptive_counts.to_csv(OUTPUT_DIR / "lifecycle_descriptive_counts.csv", index=False)

count_test_results = pd.DataFrame([
    mann_whitney_cliff(process_df, feature) for feature in all_count_features
]).dropna(how="all")
count_test_results["family"] = count_test_results["variable"].map(feature_to_family)
count_test_results["effect_size"] = count_test_results["cliffs_delta"].map(cliff_interpretation)
count_test_results = add_holm(count_test_results)
count_test_results.to_csv(OUTPUT_DIR / "count_mannwhitney_results.csv", index=False)

dispersion_rows = []
for feature in all_count_features:
    values = process_df[feature].dropna()
    mean, variance = values.mean(), values.var()
    dispersion_rows.append({
        "variable": feature, "family": feature_to_family[feature],
        "mean": mean, "variance": variance,
        "variance_mean_ratio": variance / mean if mean > 0 else np.nan,
        "zero_percentage": 100 * values.eq(0).mean(),
    })
dispersion_df = pd.DataFrame(dispersion_rows)
dispersion_df.to_csv(OUTPUT_DIR / "count_dispersion.csv", index=False)

binary_source_features = [feature for feature in [
    "description_change_count", "story_point_change_count", "assignee_change_count",
    "team_change_count", "sprint_change_count", "blocked_count", "reopen_count",
    "review_return_count", "testing_return_count", "backward_transition_count",
    "rework_entry_count",
] if feature in process_df.columns]
binary_summary_rows = []
binary_test_rows = []
for feature in binary_source_features:
    binary_col = f"ever_{feature}"
    process_df[binary_col] = process_df[feature].fillna(0).gt(0).astype(int)
    satd_rate = process_df.loc[process_df["is_satd"] == 1, binary_col].mean()
    non_rate = process_df.loc[process_df["is_satd"] == 0, binary_col].mean()
    binary_summary_rows.append({
        "event": feature, "family": feature_to_family.get(feature),
        "satd_percentage": 100 * satd_rate, "non_satd_percentage": 100 * non_rate,
        "absolute_difference_pp": 100 * (satd_rate - non_rate),
        "risk_ratio": satd_rate / non_rate if non_rate > 0 else np.nan,
    })
    table = pd.crosstab(process_df["is_satd"], process_df[binary_col]).reindex(
        index=[0, 1], columns=[0, 1], fill_value=0
    )
    chi2, chi_p, _, expected = chi2_contingency(table)
    odds_ratio, fisher_p = fisher_exact(table)
    use_fisher = bool((expected < 5).any())
    binary_test_rows.append({
        "variable": binary_col, "event": feature, "family": feature_to_family.get(feature),
        "chi_square": chi2, "cramers_v": np.sqrt(chi2 / table.to_numpy().sum()),
        "odds_ratio_unadjusted": odds_ratio,
        "p_value": fisher_p if use_fisher else chi_p,
        "test": "Fisher exact" if use_fisher else "Chi-square", "n": table.to_numpy().sum(),
    })
binary_summary = pd.DataFrame(binary_summary_rows).sort_values("absolute_difference_pp", ascending=False)
binary_test_results = add_holm(pd.DataFrame(binary_test_rows))
binary_summary.to_csv(OUTPUT_DIR / "binary_event_prevalence.csv", index=False)
binary_test_results.to_csv(OUTPUT_DIR / "binary_event_tests.csv", index=False)

print("Count differences ranked by absolute Cliff's delta:")
print(count_test_results.sort_values("cliffs_delta", key=abs, ascending=False)[[
    "family", "variable", "satd_median", "non_satd_median", "cliffs_delta",
    "effect_size", "p_holm",
]].to_string(index=False))
print("\nBinary event prevalence:")
print(binary_summary.to_string(index=False))


## 4. Adjusted event and count associations

Models adjust for issue type, priority, project-relative Story Point complexity,
and missing estimation, with exchangeable/project-clustered inference. The process
cohort contains only three projects; therefore these adjusted estimates are
explicitly treated as exploratory regardless of nominal p-values.


In [ ]:
df["story_point"] = pd.to_numeric(df["story_point"], errors="coerce")
df["project_sp_percentile"] = df.groupby("project_id")["story_point"].rank(pct=True, na_option="keep")
df["story_point_missing"] = (df["story_point"].isna() | df["story_point"].le(0)).astype(int)
df["project_sp_percentile_filled"] = df["project_sp_percentile"].fillna(0.5)
covariate_columns = [
    "priority_group", "issue_type_group", "project_sp_percentile_filled",
    "story_point_missing",
]
process_df = process_df.drop(columns=[
    col for col in ["project_sp_percentile", "story_point_missing", "project_sp_percentile_filled"]
    if col in process_df.columns
]).merge(
    df[["issue_id", "project_sp_percentile", "story_point_missing", "project_sp_percentile_filled"]],
    on="issue_id", how="left", validate="one_to_one",
)
priority_ref = "MEDIUM" if "MEDIUM" in set(df["priority_group"].dropna()) else df["priority_group"].mode()[0]
type_ref = "TECHNICAL_WORK" if "TECHNICAL_WORK" in set(df["issue_type_group"].dropna()) else df["issue_type_group"].mode()[0]
adjustment = (
    f'is_satd + C(priority_group, Treatment(reference={priority_ref!r})) + '
    f'C(issue_type_group, Treatment(reference={type_ref!r})) + '
    'project_sp_percentile_filled + story_point_missing'
)
print("References:", {"priority": priority_ref, "issue_type": type_ref})

def fit_binary_gee(data, outcome):
    columns = [outcome, "is_satd", "project_id"] + covariate_columns
    model_data = data[columns].dropna(subset=[outcome, "project_id", "priority_group", "issue_type_group"]).copy()
    if model_data[outcome].nunique() < 2 or model_data.groupby("is_satd")[outcome].sum().min() == 0:
        raise ValueError("outcome is constant or absent in one SATD group")
    result = smf.gee(
        formula=f"{outcome} ~ {adjustment}", groups="project_id", data=model_data,
        family=sm.families.Binomial(), cov_struct=sm.cov_struct.Exchangeable(),
    ).fit()
    beta = result.params["is_satd"]
    ci = result.conf_int().loc["is_satd"]
    return {
        "outcome": outcome, "adjusted_OR": np.exp(beta),
        "ci_lower": np.exp(ci.iloc[0]), "ci_upper": np.exp(ci.iloc[1]),
        "p_value": result.pvalues["is_satd"], "n": len(model_data),
        "project_clusters": model_data["project_id"].nunique(),
    }

def fit_negative_binomial(data, outcome):
    columns = [outcome, "is_satd", "project_id"] + covariate_columns
    model_data = data[columns].dropna(subset=["project_id", "priority_group", "issue_type_group"]).copy()
    model_data[outcome] = pd.to_numeric(model_data[outcome], errors="coerce")
    model_data = model_data[model_data[outcome].notna() & model_data[outcome].ge(0)]
    if model_data[outcome].sum() == 0:
        raise ValueError("all counts are zero")
    result = smf.negativebinomial(
        formula=f"{outcome} ~ {adjustment}", data=model_data
    ).fit(disp=False, maxiter=300, cov_type="cluster", cov_kwds={"groups": model_data["project_id"]})
    beta = result.params["is_satd"]
    ci = result.conf_int().loc["is_satd"]
    return {
        "outcome": outcome, "adjusted_IRR": np.exp(beta),
        "ci_lower": np.exp(ci.iloc[0]), "ci_upper": np.exp(ci.iloc[1]),
        "p_value": result.pvalues["is_satd"], "n": len(model_data),
        "project_clusters": model_data["project_id"].nunique(),
        "converged": bool(result.mle_retvals.get("converged", False)),
    }

model_failures = []
binary_gee_rows = []
for feature in binary_source_features:
    outcome = f"ever_{feature}"
    try:
        binary_gee_rows.append(fit_binary_gee(process_df, outcome))
    except Exception as exc:
        model_failures.append({"model": "binary GEE", "outcome": outcome, "error": str(exc)})
binary_gee_results = add_holm(pd.DataFrame(binary_gee_rows)) if binary_gee_rows else pd.DataFrame()

preferred_counts = [feature for feature in [
    "status_transition_count", "scope_change_count", "estimation_change_count",
    "ownership_change_count", "description_change_count", "story_point_change_count",
    "assignee_change_count", "sprint_change_count", "blocked_count", "reopen_count",
    "review_return_count", "testing_return_count", "backward_transition_count",
] if feature in process_df.columns]
nb_rows = []
for outcome in preferred_counts:
    try:
        nb_rows.append(fit_negative_binomial(process_df, outcome))
    except Exception as exc:
        model_failures.append({"model": "negative binomial", "outcome": outcome, "error": str(exc)})
nb_results = add_holm(pd.DataFrame(nb_rows)) if nb_rows else pd.DataFrame()

binary_gee_results.to_csv(OUTPUT_DIR / "adjusted_binary_lifecycle_models.csv", index=False)
nb_results.to_csv(OUTPUT_DIR / "adjusted_negative_binomial_results.csv", index=False)
pd.DataFrame(model_failures).to_csv(OUTPUT_DIR / "model_failures.csv", index=False)
print("Adjusted binary event models:")
print(binary_gee_results.to_string(index=False) if len(binary_gee_results) else "None estimable")
print("\nAdjusted count models:")
print(nb_results.to_string(index=False) if len(nb_results) else "None estimable")
if model_failures:
    print("\nModel failures:")
    print(pd.DataFrame(model_failures).to_string(index=False))


## 5. Overall lifecycle and effort duration

`lifecycle_days` is an observed issue lifespan and includes unresolved/censored
issues; it is not interpreted as completed resolution time. Durations containing
structural zeros are reported both for all issues and conditionally among positive
observations. Gamma models use only positive durations.


In [ ]:
overall_duration_features = [feature for feature in [
    "lifecycle_days", "resolution_time_minutes", "in_progress_minutes", "total_effort_minutes"
] if feature in df.columns]
for feature in overall_duration_features:
    df[feature] = pd.to_numeric(df[feature], errors="coerce")

duration_rows = []
for feature in overall_duration_features:
    result_all = mann_whitney_cliff(df, feature)
    if result_all:
        result_all["population"] = "all observed values"
        duration_rows.append(result_all)
    positive = df[df[feature] > 0]
    result_positive = mann_whitney_cliff(positive, feature)
    if result_positive:
        result_positive["population"] = "positive values only"
        duration_rows.append(result_positive)
duration_test_results = pd.DataFrame(duration_rows)
duration_test_results["effect_size"] = duration_test_results["cliffs_delta"].map(cliff_interpretation)
duration_test_results = add_holm(duration_test_results)
duration_test_results.to_csv(OUTPUT_DIR / "lifecycle_duration_tests.csv", index=False)

def fit_gamma_gee(data, outcome):
    columns = [outcome, "is_satd", "project_id"] + covariate_columns
    model_data = data[columns].dropna(subset=["project_id", "priority_group", "issue_type_group"]).copy()
    model_data[outcome] = pd.to_numeric(model_data[outcome], errors="coerce")
    model_data = model_data[model_data[outcome] > 0]
    if len(model_data) == 0 or model_data["is_satd"].nunique() < 2:
        raise ValueError("insufficient positive observations")
    result = smf.gee(
        formula=f"{outcome} ~ {adjustment}", groups="project_id", data=model_data,
        family=sm.families.Gamma(link=sm.families.links.Log()),
        cov_struct=sm.cov_struct.Exchangeable(),
    ).fit(maxiter=200)
    beta = result.params["is_satd"]
    ci = result.conf_int().loc["is_satd"]
    return {
        "outcome": outcome, "adjusted_ratio": np.exp(beta),
        "ci_lower": np.exp(ci.iloc[0]), "ci_upper": np.exp(ci.iloc[1]),
        "p_value": result.pvalues["is_satd"], "n": len(model_data),
        "project_clusters": model_data["project_id"].nunique(),
    }

overall_gamma_rows = []
for outcome in overall_duration_features:
    try:
        overall_gamma_rows.append(fit_gamma_gee(df, outcome))
    except Exception as exc:
        model_failures.append({"model": "overall Gamma GEE", "outcome": outcome, "error": str(exc)})
overall_gamma_results = add_holm(pd.DataFrame(overall_gamma_rows)) if overall_gamma_rows else pd.DataFrame()
overall_gamma_results.to_csv(OUTPUT_DIR / "adjusted_overall_duration_models.csv", index=False)
pd.DataFrame(model_failures).to_csv(OUTPUT_DIR / "model_failures.csv", index=False)
print(duration_test_results[[
    "variable", "population", "satd_n", "non_satd_n", "satd_median",
    "non_satd_median", "cliffs_delta", "effect_size", "p_holm",
]].to_string(index=False))
print("\nAdjusted positive-duration ratios:")
print(overall_gamma_results.to_string(index=False))


## 6. Stage entry and duration

Entry prevalence uses every observable-history issue as the denominator. Duration
comparisons use the stage-duration rows and separately report positive-duration
comparisons. Adjusted stage estimates have only three project clusters and are
exploratory.


In [ ]:
observed_stages = sorted(stage_df["lifecycle_stage"].dropna().unique())
entry_rows = []
for stage in observed_stages:
    entered_ids = set(stage_df.loc[stage_df["lifecycle_stage"] == stage, "issue_id"])
    indicator = process_df["issue_id"].isin(entered_ids)
    satd_rate = indicator[process_df["is_satd"].eq(1)].mean()
    non_rate = indicator[process_df["is_satd"].eq(0)].mean()
    table = pd.crosstab(process_df["is_satd"], indicator.astype(int)).reindex(
        index=[0, 1], columns=[0, 1], fill_value=0
    )
    odds_ratio, p_value = fisher_exact(table)
    entry_rows.append({
        "lifecycle_stage": stage, "satd_entry_percentage": 100 * satd_rate,
        "non_satd_entry_percentage": 100 * non_rate,
        "difference_pp": 100 * (satd_rate - non_rate),
        "risk_ratio": satd_rate / non_rate if non_rate else np.nan,
        "odds_ratio": odds_ratio, "p_value": p_value,
    })
stage_entry_results = add_holm(pd.DataFrame(entry_rows))
stage_entry_results.to_csv(OUTPUT_DIR / "stage_entry_prevalence.csv", index=False)

stage_duration_rows = []
positive_stage_duration_rows = []
for stage, data in stage_df.groupby("lifecycle_stage"):
    temp = data.rename(columns={"observed_minutes_in_stage": "duration"}).copy()
    result = mann_whitney_cliff(temp, "duration")
    if result:
        result["lifecycle_stage"] = stage
        stage_duration_rows.append(result)
    positive = temp[temp["duration"] > 0]
    result_positive = mann_whitney_cliff(positive, "duration")
    if result_positive:
        result_positive["lifecycle_stage"] = stage
        positive_stage_duration_rows.append(result_positive)

stage_duration_results = pd.DataFrame(stage_duration_rows)
positive_stage_duration_results = pd.DataFrame(positive_stage_duration_rows)
for results in [stage_duration_results, positive_stage_duration_results]:
    if len(results):
        results["effect_size"] = results["cliffs_delta"].map(cliff_interpretation)
        corrected = add_holm(results)
        results["p_holm"] = corrected["p_holm"]
        results["significant_holm"] = corrected["significant_holm"]
stage_duration_results.to_csv(OUTPUT_DIR / "stage_duration_tests.csv", index=False)
positive_stage_duration_results.to_csv(OUTPUT_DIR / "positive_stage_duration_results.csv", index=False)

stage_covariates = df[["issue_id"] + covariate_columns].copy()
stage_model_df = stage_df.merge(stage_covariates, on="issue_id", how="left", validate="many_to_one")
stage_gamma_rows = []
for stage, data in stage_model_df.groupby("lifecycle_stage"):
    temp = data.rename(columns={"observed_minutes_in_stage": "stage_minutes"}).copy()
    try:
        result = fit_gamma_gee(temp, "stage_minutes")
        result["lifecycle_stage"] = stage
        stage_gamma_rows.append(result)
    except Exception as exc:
        model_failures.append({"model": "stage Gamma GEE", "outcome": stage, "error": str(exc)})
stage_gamma_results = add_holm(pd.DataFrame(stage_gamma_rows)) if stage_gamma_rows else pd.DataFrame()
stage_gamma_results.to_csv(OUTPUT_DIR / "adjusted_stage_duration_models.csv", index=False)
pd.DataFrame(model_failures).to_csv(OUTPUT_DIR / "model_failures.csv", index=False)

print("Stage-entry prevalence:")
print(stage_entry_results.to_string(index=False))
print("\nPositive stage-duration comparisons:")
print(positive_stage_duration_results[[
    "lifecycle_stage", "satd_n", "non_satd_n", "satd_median", "non_satd_median",
    "cliffs_delta", "effect_size", "p_holm",
]].to_string(index=False))
print("\nExploratory adjusted stage-duration ratios:")
print(stage_gamma_results.to_string(index=False) if len(stage_gamma_results) else "None estimable")


## 7. Workflow transition probabilities

Event-level conditional probabilities reproduce the lifecycle transition matrix.
Because repeated transitions from one issue are not independent, formal association
tests are performed on issue-level “ever experienced this transition” indicators.


In [ ]:
transition_counts = (
    transitions.groupby(["is_satd", "from_stage", "to_stage"]).size()
    .reset_index(name="transition_count")
)
transition_counts["source_total"] = transition_counts.groupby(
    ["is_satd", "from_stage"]
)["transition_count"].transform("sum")
transition_counts["transition_probability"] = (
    transition_counts["transition_count"] / transition_counts["source_total"]
)
transition_counts.to_csv(OUTPUT_DIR / "transition_probabilities.csv", index=False)

transition_comparison = transition_counts.pivot_table(
    index=["from_stage", "to_stage"], columns="is_satd",
    values="transition_probability", fill_value=0,
).reset_index()
transition_comparison.columns.name = None
for label in [0, 1]:
    if label not in transition_comparison.columns:
        transition_comparison[label] = 0.0
transition_comparison = transition_comparison.rename(
    columns={0: "non_satd_probability", 1: "satd_probability"}
)
transition_comparison["probability_difference"] = (
    transition_comparison["satd_probability"] - transition_comparison["non_satd_probability"]
)
transition_comparison["probability_ratio"] = np.where(
    transition_comparison["non_satd_probability"] > 0,
    transition_comparison["satd_probability"] / transition_comparison["non_satd_probability"],
    np.nan,
)
transition_comparison.to_csv(OUTPUT_DIR / "transition_probability_comparison.csv", index=False)
transition_comparison.sort_values("probability_difference", ascending=False).head(20).to_csv(
    OUTPUT_DIR / "top_satd_enriched_transitions.csv", index=False
)

unique_issue_transitions = transitions[["issue_id", "from_stage", "to_stage"]].drop_duplicates()
issue_transition_rows = []
n_satd = int(process_df["is_satd"].sum())
n_non = int((process_df["is_satd"] == 0).sum())
for (from_stage, to_stage), pair_data in unique_issue_transitions.groupby(["from_stage", "to_stage"]):
    ids = set(pair_data["issue_id"])
    satd_yes = int(process_df.loc[process_df["is_satd"] == 1, "issue_id"].isin(ids).sum())
    non_yes = int(process_df.loc[process_df["is_satd"] == 0, "issue_id"].isin(ids).sum())
    table = np.array([[non_yes, n_non - non_yes], [satd_yes, n_satd - satd_yes]])
    odds_ratio, p_value = fisher_exact(table)
    satd_rate, non_rate = satd_yes / n_satd, non_yes / n_non
    issue_transition_rows.append({
        "from_stage": from_stage, "to_stage": to_stage,
        "satd_issues": satd_yes, "non_satd_issues": non_yes,
        "satd_issue_percentage": 100 * satd_rate,
        "non_satd_issue_percentage": 100 * non_rate,
        "difference_pp": 100 * (satd_rate - non_rate),
        "odds_ratio": odds_ratio, "p_value": p_value,
    })
issue_transition_results = add_holm(pd.DataFrame(issue_transition_rows))
issue_transition_results.to_csv(OUTPUT_DIR / "issue_level_transition_tests.csv", index=False)

important_pairs = [
    ("REVIEW", "DEVELOPMENT"), ("TESTING", "DEVELOPMENT"),
    ("DEVELOPMENT", "BLOCKED_WAITING"), ("REVIEW", "REWORK"),
    ("TESTING", "REWORK"), ("REWORK", "DEVELOPMENT"),
]
important_transition_results = issue_transition_results[
    issue_transition_results.apply(
        lambda row: (row["from_stage"], row["to_stage"]) in important_pairs, axis=1
    )
].copy()
important_transition_results.to_csv(OUTPUT_DIR / "important_transition_results.csv", index=False)
print("Largest issue-level transition prevalence differences (minimum 20 exposed issues):")
print(issue_transition_results[
    (issue_transition_results["satd_issues"] + issue_transition_results["non_satd_issues"]) >= 20
].sort_values("difference_pp", ascending=False).head(15).to_string(index=False))
print("\nPre-specified transitions:")
print(important_transition_results.to_string(index=False))


In [ ]:
preferred_stage_order = [
    "BACKLOG_INTAKE", "PLANNING", "DEVELOPMENT", "REVIEW", "TESTING",
    "BLOCKED_WAITING", "REWORK", "RELEASE", "COMPLETED", "OTHER", "UNKNOWN",
]
observed = set(transitions["from_stage"]) | set(transitions["to_stage"])
stage_order = [stage for stage in preferred_stage_order if stage in observed]

def transition_matrix(data, satd_value):
    matrix = pd.crosstab(
        data.loc[data["is_satd"] == satd_value, "from_stage"],
        data.loc[data["is_satd"] == satd_value, "to_stage"], normalize="index",
    )
    return matrix.reindex(index=stage_order, columns=stage_order, fill_value=0)

satd_matrix = transition_matrix(transitions, 1)
non_satd_matrix = transition_matrix(transitions, 0)
difference_matrix = satd_matrix - non_satd_matrix
satd_matrix.to_csv(OUTPUT_DIR / "transition_matrix_satd.csv")
non_satd_matrix.to_csv(OUTPUT_DIR / "transition_matrix_non_satd.csv")
difference_matrix.to_csv(OUTPUT_DIR / "transition_probability_difference.csv")

for matrix, title, filename, cmap, symmetric in [
    (non_satd_matrix, "Lifecycle transition matrix - Non-SATD", "transition_matrix_non_satd.png", "viridis", False),
    (satd_matrix, "Lifecycle transition matrix - SATD", "transition_matrix_satd.png", "viridis", False),
    (difference_matrix, "Transition probability difference: SATD - Non-SATD", "transition_probability_difference.png", "coolwarm", True),
]:
    fig, ax = plt.subplots(figsize=(11, 9))
    kwargs = {}
    if symmetric:
        limit = max(abs(matrix.to_numpy().min()), abs(matrix.to_numpy().max()))
        kwargs.update(vmin=-limit, vmax=limit)
    image = ax.imshow(matrix.to_numpy(), aspect="auto", cmap=cmap, **kwargs)
    ax.set_xticks(range(len(stage_order)), labels=stage_order, rotation=45, ha="right")
    ax.set_yticks(range(len(stage_order)), labels=stage_order)
    ax.set(xlabel="To stage", ylabel="From stage", title=title)
    fig.colorbar(image, ax=ax, label="Probability" if not symmetric else "Probability difference")
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / filename, dpi=300)
    plt.close(fig)


## 8. Workflow paths, path complexity, and rework intensity


In [ ]:
if "workflow_path" in process_df.columns:
    satd_paths = process_df.loc[process_df["is_satd"] == 1, "workflow_path"].dropna().value_counts().head(20).rename_axis("workflow_path").reset_index(name="frequency")
    non_satd_paths = process_df.loc[process_df["is_satd"] == 0, "workflow_path"].dropna().value_counts().head(20).rename_axis("workflow_path").reset_index(name="frequency")
    satd_paths.to_csv(OUTPUT_DIR / "top_satd_workflow_paths.csv", index=False)
    non_satd_paths.to_csv(OUTPUT_DIR / "top_non_satd_workflow_paths.csv", index=False)

    def count_unique_stages(path):
        if pd.isna(path): return np.nan
        return len({value.strip() for value in str(path).split("->") if value.strip()})
    process_df["unique_stage_count"] = process_df["workflow_path"].map(count_unique_stages)
else:
    process_df["unique_stage_count"] = np.nan

rework_components = [feature for feature in [
    "reopen_count", "review_return_count", "testing_return_count", "backward_transition_count"
] if feature in process_df.columns]
process_df["rework_intensity"] = process_df[rework_components].fillna(0).sum(axis=1)
composite_rows = []
for feature in ["unique_stage_count", "rework_intensity"]:
    result = mann_whitney_cliff(process_df, feature)
    if result:
        result["effect_size"] = cliff_interpretation(result["cliffs_delta"])
        composite_rows.append(result)
workflow_composite_results = add_holm(pd.DataFrame(composite_rows))
workflow_composite_results.to_csv(OUTPUT_DIR / "workflow_composite_results.csv", index=False)
print(workflow_composite_results.to_string(index=False))


## 9. Secondary comparison across SATD types

Kruskal-Wallis tests are restricted to SATD types with at least 20 issues in the
observable-history cohort and are Holm-adjusted. This remains secondary to the
binary SATD/non-SATD RQ2 analysis.


In [ ]:
satd_only = process_df[process_df["is_satd"] == 1].copy()
satd_type_counts = satd_only["satd_type"].value_counts(dropna=False).rename_axis("satd_type").reset_index(name="issues")
satd_type_counts.to_csv(OUTPUT_DIR / "satd_type_counts_process_cohort.csv", index=False)

satd_type_features = [feature for feature in [
    "scope_change_count", "estimation_change_count", "ownership_change_count",
    "status_transition_count", "rework_intensity", "lifecycle_days",
] if feature in satd_only.columns]
satd_type_rows = []
for feature in satd_type_features:
    samples, labels = [], []
    for label, group in satd_only.groupby("satd_type"):
        values = pd.to_numeric(group[feature], errors="coerce").dropna()
        if len(values) >= MIN_SATD_TYPE_GROUP:
            labels.append(str(label)); samples.append(values.to_numpy())
    if len(samples) >= 2:
        statistic, p_value = kruskal(*samples)
        satd_type_rows.append({
            "variable": feature, "groups": " | ".join(labels),
            "group_count": len(samples), "kruskal_h": statistic, "p_value": p_value,
        })
satd_type_results = add_holm(pd.DataFrame(satd_type_rows)) if satd_type_rows else pd.DataFrame()
satd_type_results.to_csv(OUTPUT_DIR / "satd_type_kruskal_results.csv", index=False)
print(satd_type_counts.to_string(index=False))
print("\nSATD-type tests:")
print(satd_type_results.to_string(index=False) if len(satd_type_results) else "Insufficient groups")


## 10. Main figures and consolidated evidence table


In [ ]:
plotted = count_test_results.sort_values("cliffs_delta", key=abs).copy()
colors = plotted["family"].map({
    "Scope": "#4C78A8", "Estimation": "#F58518",
    "Ownership": "#54A24B", "Workflow": "#E45756",
})
fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(plotted["variable"], plotted["cliffs_delta"], color=colors)
ax.axvline(0, color="black", linewidth=0.8)
ax.axvline(0.147, color="gray", linestyle="--", linewidth=0.8)
ax.axvline(-0.147, color="gray", linestyle="--", linewidth=0.8)
ax.set(xlabel="Cliff's delta (positive = larger for SATD)", title="Lifecycle count differences")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "lifecycle_count_effect_sizes.png", dpi=300)
plt.close(fig)

if len(positive_stage_duration_results):
    temp = positive_stage_duration_results.sort_values("cliffs_delta")
    fig, ax = plt.subplots(figsize=(9, 6))
    ax.barh(temp["lifecycle_stage"], temp["cliffs_delta"], color="#72B7B2")
    ax.axvline(0, color="black", linewidth=0.8)
    ax.axvline(0.147, color="gray", linestyle="--", linewidth=0.8)
    ax.axvline(-0.147, color="gray", linestyle="--", linewidth=0.8)
    ax.set(xlabel="Cliff's delta (positive = longer for SATD)", title="Positive stage-duration differences")
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "stage_duration_effect_sizes.png", dpi=300)
    plt.close(fig)

rq2_main_results = count_test_results[[
    "family", "variable", "satd_median", "non_satd_median", "cliffs_delta",
    "effect_size", "p_holm", "significant_holm",
]].copy()
if len(nb_results):
    rq2_main_results = rq2_main_results.merge(
        nb_results[["outcome", "adjusted_IRR", "ci_lower", "ci_upper", "p_holm"]],
        left_on="variable", right_on="outcome", how="left",
        suffixes=("_unadjusted", "_adjusted"),
    )
rq2_main_results.to_csv(OUTPUT_DIR / "rq2_main_results.csv", index=False)

family_summary = (
    count_test_results.assign(abs_delta=count_test_results["cliffs_delta"].abs())
    .sort_values(["family", "abs_delta"], ascending=[True, False])
    .groupby("family", as_index=False).first()
)
family_summary.to_csv(OUTPUT_DIR / "rq2_family_summary.csv", index=False)
print(family_summary[[
    "family", "variable", "cliffs_delta", "effect_size", "p_holm"
]].to_string(index=False))


## 11. Data-driven answer to RQ2

The cell below writes the answer from the observed effect sizes, adjusted ratios,
duration results, and coverage audit. Statistical significance alone is not treated
as practical importance.


In [ ]:
def fmt_p(value):
    return "NA" if pd.isna(value) else ("<0.001" if value < 0.001 else f"{value:.3f}")

answer_lines = [
    "RQ2 answer",
    "",
    f"The observable-history comparison included {len(process_df):,} issues "
    f"({int(process_df['is_satd'].sum()):,} SATD) from {process_df['project_id'].nunique()} projects. "
    f"Overall-duration comparisons included {len(df):,} issues from {df['project_id'].nunique()} projects.",
    "",
]
for family in ["Scope", "Estimation", "Ownership", "Workflow"]:
    family_rows = count_test_results[count_test_results["family"] == family].copy()
    strongest = family_rows.loc[family_rows["cliffs_delta"].abs().idxmax()]
    direction = "more" if strongest["cliffs_delta"] > 0 else "fewer"
    answer_lines.append(
        f"{family}: the largest count difference was {strongest['variable']} "
        f"(SATD median {strongest['satd_median']:.2f}, non-SATD median "
        f"{strongest['non_satd_median']:.2f}; Cliff's delta {strongest['cliffs_delta']:+.3f}, "
        f"{strongest['effect_size'].lower()}; Holm p {fmt_p(strongest['p_holm'])}), "
        f"indicating {direction} observed activity for SATD issues."
    )

life_positive = duration_test_results[
    (duration_test_results["variable"] == "lifecycle_days") &
    (duration_test_results["population"] == "positive values only")
]
if len(life_positive):
    row = life_positive.iloc[0]
    answer_lines.append(
        f"Temporal evolution: SATD issues had a substantially longer observed lifespan "
        f"(median {row['satd_median']:.1f} versus {row['non_satd_median']:.1f} days; "
        f"Cliff's delta {row['cliffs_delta']:+.3f}, {row['effect_size'].lower()}; "
        f"Holm p {fmt_p(row['p_holm'])})."
    )

significant_stage = positive_stage_duration_results[
    positive_stage_duration_results.get("significant_holm", False)
] if len(positive_stage_duration_results) else pd.DataFrame()
if len(significant_stage):
    row = significant_stage.loc[significant_stage["cliffs_delta"].abs().idxmax()]
    answer_lines.append(
        f"Among entered stages, the strongest Holm-significant duration difference was "
        f"{row['lifecycle_stage']} (Cliff's delta {row['cliffs_delta']:+.3f})."
    )
else:
    answer_lines.append(
        "No positive stage-duration difference remained significant after Holm correction."
    )

reliable_transition = issue_transition_results[
    issue_transition_results["significant_holm"] &
    ((issue_transition_results["satd_issues"] + issue_transition_results["non_satd_issues"]) >= 20)
]
if len(reliable_transition):
    row = reliable_transition.loc[reliable_transition["difference_pp"].abs().idxmax()]
    answer_lines.append(
        f"The largest supported issue-level transition difference was "
        f"{row['from_stage']}->{row['to_stage']} ({row['satd_issue_percentage']:.1f}% of SATD "
        f"versus {row['non_satd_issue_percentage']:.1f}% of non-SATD issues; "
        f"difference {row['difference_pp']:+.1f} percentage points)."
    )
else:
    answer_lines.append("No common issue-level transition survived Holm correction.")

meaningful_counts = count_test_results[
    count_test_results["significant_holm"] & count_test_results["effect_size"].ne("Negligible")
]
if len(meaningful_counts) >= 4:
    conclusion = (
        "Overall, SATD issues show a consistently more unstable lifecycle across several dimensions."
    )
elif len(meaningful_counts) >= 1:
    supported_families = ", ".join(sorted(meaningful_counts["family"].unique()))
    conclusion = (
        "Overall, lifecycle differences are partial rather than universal. The practically clearest "
        f"count differences occur in {supported_families}, while most individual count effects are negligible."
    )
else:
    conclusion = (
        "Overall, several comparisons may be statistically detectable, but count-effect magnitudes are "
        "generally negligible, providing limited evidence of substantially different process trajectories."
    )
answer_lines.extend([
    "", conclusion,
    "The strongest broad difference is temporal: SATD-labelled issues remain observable much longer and, "
    "among issues with recorded effort, tend to accumulate more effort. This is an association, not evidence "
    "that longer duration or instability causes SATD.",
    "Because detailed histories cover only three projects and only issues with at least one recorded transition, "
    "the process and transition findings are exploratory and do not support a dataset-wide causal claim.",
])
rq2_answer = "\n".join(answer_lines)
(OUTPUT_DIR / "rq2_answer.txt").write_text(rq2_answer, encoding="utf-8")
print(rq2_answer)
print("\nArtifacts written to", OUTPUT_DIR.resolve())


## 12. Censoring-aware time-to-resolution analysis

The earlier `lifecycle_days` comparison is descriptive and cannot properly handle unresolved issues. Here, a resolution timestamp defines an observed event; unresolved issues are right-censored at the last observed timestamp in their project. Kaplan–Meier estimates and a project-stratified Cox model are therefore used to compare time to recorded resolution. A hazard ratio below 1 means a lower instantaneous resolution rate and hence a tendency to remain unresolved longer.


In [ ]:
from statsmodels.duration.hazard_regression import PHReg
from statsmodels.duration.survfunc import SurvfuncRight, survdiff

survival_df = df.copy()
for column in ["creation_ts", "resolution_ts", "last_updated_ts"]:
    survival_df[column] = pd.to_datetime(survival_df[column], errors="coerce")

survival_df["observed_end_ts"] = survival_df[["resolution_ts", "last_updated_ts"]].max(axis=1)
project_end = survival_df.groupby("project_id")["observed_end_ts"].max()
survival_df["project_end_ts"] = survival_df["project_id"].map(project_end)
survival_df["resolution_event"] = survival_df["resolution_ts"].notna().astype(int)
survival_df["analysis_end_ts"] = survival_df["resolution_ts"].where(
    survival_df["resolution_event"].eq(1), survival_df["project_end_ts"]
)
survival_df["time_to_resolution_days"] = (
    survival_df["analysis_end_ts"] - survival_df["creation_ts"]
).dt.total_seconds() / 86400.0
survival_df = survival_df[
    survival_df["time_to_resolution_days"].notna()
    & survival_df["time_to_resolution_days"].gt(0)
].copy()

km_rows = []
km_models = {}
for satd_value, label in [(0, "Non-SATD"), (1, "SATD")]:
    group = survival_df[survival_df["is_satd"].eq(satd_value)]
    km = SurvfuncRight(
        group["time_to_resolution_days"].to_numpy(),
        group["resolution_event"].to_numpy(),
    )
    km_models[label] = km
    km_rows.append({
        "group": label,
        "issues": len(group),
        "resolution_events": int(group["resolution_event"].sum()),
        "censored_issues": int(group["resolution_event"].eq(0).sum()),
        "event_percentage": 100 * group["resolution_event"].mean(),
        "km_median_days": km.quantile(0.5),
    })

logrank_chi2, logrank_p = survdiff(
    survival_df["time_to_resolution_days"].to_numpy(),
    survival_df["resolution_event"].to_numpy(),
    survival_df["is_satd"].to_numpy(),
)
survival_summary = pd.DataFrame(km_rows)
survival_summary["logrank_chi2"] = logrank_chi2
survival_summary["logrank_p_value"] = logrank_p
survival_summary.to_csv(OUTPUT_DIR / "survival_summary.csv", index=False)

# PHReg is computationally expensive at the full cohort size. Draw a fixed,
# reproducible project-by-label stratified estimation sample (up to 500 per
# stratum); Kaplan-Meier and log-rank estimates above still use the full cohort.
cox_df = (survival_df.groupby(["project_id", "is_satd"], group_keys=False)
          .apply(lambda group: group.sample(min(len(group), 500), random_state=42))
          .reset_index(drop=True))
cox_df["project_sp_percentile_survival"] = (
    cox_df["story_point"].where(cox_df["story_point"].gt(0))
    .groupby(cox_df["project_id"]).rank(pct=True)
)
cox_df["story_point_missing_survival"] = (
    cox_df["story_point"].isna() | cox_df["story_point"].le(0)
).astype(int)
cox_df["project_sp_percentile_survival"] = cox_df["project_sp_percentile_survival"].fillna(0.5)
# Keep the pre-specified adjustment compact: this avoids unstable, very slow
# dummy-variable estimation on more than 200,000 observations. Project-specific
# baseline hazards are handled by stratifying the Cox model by project.
cox_covariates = cox_df[[
    "is_satd", "project_sp_percentile_survival", "story_point_missing_survival",
]].copy()
cox_covariates = cox_covariates.astype(float)
cox_model = PHReg(
    cox_df["time_to_resolution_days"].to_numpy(),
    cox_covariates.to_numpy(),
    status=cox_df["resolution_event"].to_numpy(),
    ties="breslow",
    strata=cox_df["project_id"].to_numpy(),
)
cox_result = cox_model.fit()
cox_ci = np.asarray(cox_result.conf_int())
cox_results = pd.DataFrame({
    "term": cox_covariates.columns,
    "coefficient": np.asarray(cox_result.params),
    "hazard_ratio": np.exp(np.asarray(cox_result.params)),
    "ci_lower": np.exp(cox_ci[:, 0]),
    "ci_upper": np.exp(cox_ci[:, 1]),
    "p_value": np.asarray(cox_result.pvalues),
    "estimation_sample_n": len(cox_df),
})
cox_results.to_csv(OUTPUT_DIR / "cox_time_to_resolution.csv", index=False)

plt.figure(figsize=(8, 6))
for label, km in km_models.items():
    plt.step(km.surv_times, km.surv_prob, where="post", label=label)
plt.xlabel("Days since issue creation")
plt.ylabel("Probability of remaining unresolved")
plt.title("Time to Recorded Resolution by Eventual SATD Classification")
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "time_to_resolution_km.png", dpi=300)
plt.show()

print(survival_summary.round(3).to_string(index=False))
print("\nSATD coefficient from adjusted Cox model:")
print(cox_results[cox_results["term"].eq("is_satd")].round(4).to_string(index=False))
